# Train the Market-1501 Strong Baseline on a Colab GPU

This notebook reproduces [`Secondsight`](https://github.com/vardhjain/Secondsight)
end to end on a **free Colab GPU**:

1. Install the package
2. Download Market-1501
3. Train the ResNet-50 + BNNeck strong baseline (~40 min on a T4)
4. Evaluate (CMC / mAP + k-reciprocal re-ranking)
5. Generate Grad-CAM and ranked-result visualizations
6. Download the trained checkpoint

> **Before you start:** enable the GPU via *Runtime -> Change runtime type -> T4 GPU*.


## 1. Verify the GPU


In [ ]:
!nvidia-smi


## 2. Get the code and install it

This cell finds the project (cloning it from GitHub if it isn't already in
Colab), installs it with `pip install -e .`, and puts `src/` on the path so
`import reid` works in this kernel **and** in the `!python scripts/...` cells
below. If the repo isn't on GitHub yet, upload/unzip it to
`/content/Secondsight` first and re-run this cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vardhjain/Secondsight.git"
REPO_DIR = "/content/Secondsight"


def _find_root():
    """Return the project root (dir with pyproject.toml + src/reid), or None."""
    for candidate in (REPO_DIR, os.getcwd(), "/content"):
        root = Path(candidate)
        if (root / "pyproject.toml").is_file() and (root / "src" / "reid").is_dir():
            return root
    return None


root = _find_root()
if root is None:
    print("Project not found locally; cloning from GitHub...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=False)
    root = _find_root()

assert root is not None, (
    "Could not find the project (need pyproject.toml + src/reid). Push the repo to "
    "GitHub, or upload/unzip it to /content/Secondsight, then re-run."
)

os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Belt and suspenders: make `reid` importable in this kernel AND in the
# `!python scripts/...` subprocesses below, regardless of the editable install.
src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")

import reid

print("Project root:", root)
print("reid", reid.__version__, "->", reid.__file__)


## 3. Download Market-1501

Pulled from Kaggle via `kagglehub`; its root is exported as `REID_DATA_ROOT`,
which every CLI reads as a fallback when `--data-root` is omitted.


In [ ]:
import os
from pathlib import Path

import kagglehub

download_path = Path(kagglehub.dataset_download("pengcw1/market-1501"))
canonical = download_path / "Market-1501-v15.09.15"
data_root = canonical if canonical.is_dir() else download_path
os.environ["REID_DATA_ROOT"] = str(data_root)
print("Market-1501 data root:", data_root)


## 4. Train the strong baseline

Checkpoints are written to `outputs/`; `best.pth` is the best-by-mAP checkpoint.
Tip: add `--max-epochs 5` for a quick smoke run before the full ~40 min train.


In [ ]:
!python scripts/train.py --config configs/market1501_strong_baseline.yaml --device cuda --output-dir outputs


## 5. Evaluate (CMC / mAP + k-reciprocal re-ranking)


In [ ]:
!python scripts/evaluate.py --config configs/market1501_strong_baseline.yaml --weights outputs/best.pth --device cuda --rerank


## 6. Visualize (Grad-CAM + ranked results)


In [ ]:
!python scripts/visualize.py --config configs/market1501_strong_baseline.yaml --weights outputs/best.pth --device cuda --output outputs/viz --num-gradcam 5


## 7. Save your results

Download the trained checkpoint, then copy the printed **mAP / Rank-1 / Rank-5**
numbers into the README *Results* table.


In [ ]:
from google.colab import files

files.download("outputs/best.pth")
